In [1]:
import pandas as pd
import numpy as np
import re
import os
import cv2
from PIL import Image
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

captions_path = "../data/captions.txt"
images_dir = "../data/Images"
processed_dir = "../data/processed"

df = pd.read_csv(captions_path)
print(df.shape)
df.head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


(40455, 2)


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


In [2]:
stop_words = set(stopwords.words('english'))

def clean_caption(text):
    text = str(text).lower()                        # lowercasing
    text = re.sub(r'[^a-z\s]', '', text)             # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()         # remove extra spaces
    tokens = word_tokenize(text)                     # tokenization
    tokens = [t for t in tokens if t not in stop_words]  # stop-word removal
    return tokens

# Test on a sample
sample = df['caption'].iloc[0]
print("Original:", sample)
print("Cleaned tokens:", clean_caption(sample))

Original: A child in a pink dress is climbing up a set of stairs in an entry way .
Cleaned tokens: ['child', 'pink', 'dress', 'climbing', 'set', 'stairs', 'entry', 'way']


In [3]:
df['tokens'] = df['caption'].apply(clean_caption)
df['cleaned_caption'] = df['tokens'].apply(lambda x: ' '.join(x))

df[['image', 'caption', 'cleaned_caption', 'tokens']].head(10)

,image,caption,cleaned_caption,tokens
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...,child pink dress climbing set stairs entry way,"[child, pink, dress, climbing, set, stairs, en..."
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .,girl going wooden building,"[girl, going, wooden, building]"
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .,little girl climbing wooden playhouse,"[little, girl, climbing, wooden, playhouse]"
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...,little girl climbing stairs playhouse,"[little, girl, climbing, stairs, playhouse]"
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...,little girl pink dress going wooden cabin,"[little, girl, pink, dress, going, wooden, cabin]"
5,1001773457_577c3a7d70.jpg,A black dog and a spotted dog are fighting,black dog spotted dog fighting,"[black, dog, spotted, dog, fighting]"
6,1001773457_577c3a7d70.jpg,A black dog and a tri-colored dog playing with...,black dog tricolored dog playing road,"[black, dog, tricolored, dog, playing, road]"
7,1001773457_577c3a7d70.jpg,A black dog and a white dog with brown spots a...,black dog white dog brown spots staring street,"[black, dog, white, dog, brown, spots, staring..."
8,1001773457_577c3a7d70.jpg,Two dogs of different breeds looking at each o...,two dogs different breeds looking road,"[two, dogs, different, breeds, looking, road]"
9,1001773457_577c3a7d70.jpg,Two dogs on pavement moving toward each other .,two dogs pavement moving toward,"[two, dogs, pavement, moving, toward]"


In [4]:
all_tokens = [token for tokens in df['tokens'] for token in tokens]
vocab_counter = Counter(all_tokens)
vocab = sorted(vocab_counter.keys())

print(f"Vocabulary size (after cleaning): {len(vocab)}")
print("Top 20 most common words:", vocab_counter.most_common(20))

# Save vocabulary to file
with open(os.path.join(processed_dir, "vocabulary.txt"), "w", encoding="utf-8") as f:
    for word in vocab:
        f.write(word + "\n")

Vocabulary size (after cleaning): 8661
Top 20 most common words: [('dog', 8136), ('man', 7265), ('two', 5638), ('white', 3940), ('black', 3832), ('boy', 3581), ('woman', 3402), ('girl', 3328), ('wearing', 3062), ('people', 2883), ('water', 2783), ('red', 2672), ('young', 2630), ('brown', 2563), ('blue', 2268), ('dogs', 2125), ('running', 2073), ('playing', 2008), ('shirt', 1806), ('standing', 1787)]


In [5]:
output_df = df[['image', 'cleaned_caption']]
output_df.to_csv(os.path.join(processed_dir, "cleaned_captions.csv"), index=False)
print("Saved cleaned_captions.csv")
output_df.head()

Saved cleaned_captions.csv


,image,cleaned_caption
0,1000268201_693b08cb0e.jpg,child pink dress climbing set stairs entry way
1,1000268201_693b08cb0e.jpg,girl going wooden building
2,1000268201_693b08cb0e.jpg,little girl climbing wooden playhouse
3,1000268201_693b08cb0e.jpg,little girl climbing stairs playhouse
4,1000268201_693b08cb0e.jpg,little girl pink dress going wooden cabin


In [6]:
IMG_SIZE = (224, 224)  # standard size for most CV/deep learning models

def preprocess_image(img_path, target_size=IMG_SIZE):
    img = cv2.imread(img_path)
    if img is None:
        return None, "Failed to load"
    
    channels = img.shape[2] if len(img.shape) == 3 else 1
    if channels != 3:
        return None, f"Unexpected channels: {channels}"
    
    img_resized = cv2.resize(img, target_size)
    img_normalized = img_resized.astype(np.float32) / 255.0  # normalize to [0,1]
    
    return img_normalized, "OK"

# Test on one image
test_img_name = df['image'].iloc[0]
test_path = os.path.join(images_dir, test_img_name)
result, status = preprocess_image(test_path)
print(status, "- Shape:", result.shape if result is not None else None)

OK - Shape: (224, 224, 3)


In [7]:

sample_size = 200  
sample_images = df['image'].unique()[:sample_size]

processed_count = 0
error_log = []

for img_name in sample_images:
    img_path = os.path.join(images_dir, img_name)
    processed_img, status = preprocess_image(img_path)
    
    if processed_img is not None:
        save_path = os.path.join(processed_dir, "images", img_name)
        # Save as uint8 for viewing (0-255), normalized version stays in-memory during preprocessing
        cv2.imwrite(save_path, (processed_img * 255).astype(np.uint8))
        processed_count += 1
    else:
        error_log.append((img_name, status))

print(f"Processed: {processed_count}/{len(sample_images)}")
print(f"Errors: {len(error_log)}")
if error_log:
    print(error_log[:10])

Processed: 200/200
Errors: 0


In [8]:
channel_issues = []
checked = 0

for img_name in df['image'].unique():
    img_path = os.path.join(images_dir, img_name)
    img = cv2.imread(img_path)
    if img is None:
        channel_issues.append((img_name, "failed to load"))
    elif len(img.shape) != 3 or img.shape[2] != 3:
        channel_issues.append((img_name, f"channels: {img.shape}"))
    checked += 1

print(f"Total images checked: {checked}")
print(f"Channel inconsistencies found: {len(channel_issues)}")
if channel_issues:
    print(channel_issues[:10])

Total images checked: 8091
Channel inconsistencies found: 0
